In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [9]:
embeddings.embed_query("Hello world")

[-0.02342152,
 0.01676572,
 0.009261323,
 -0.06383,
 -0.0026262768,
 0.0010187156,
 -0.01125684,
 0.0130036585,
 0.008751671,
 0.0012745215,
 -0.0007880878,
 -0.019086141,
 0.030971918,
 0.044264916,
 0.11137619,
 0.018025596,
 -0.0031383373,
 -0.0130702155,
 0.0054121013,
 -0.009036983,
 0.009904047,
 -0.010718052,
 0.012551899,
 0.011420718,
 -0.03375867,
 0.008877066,
 0.027795823,
 0.0037452083,
 0.029841818,
 0.019861836,
 0.018075233,
 -0.007178086,
 -0.02497957,
 0.011720823,
 -0.0073324037,
 0.0068515264,
 0.011618152,
 0.004278904,
 -0.0099426275,
 -0.009696086,
 -0.0037028084,
 0.006349137,
 -0.012098703,
 -0.015629271,
 -0.022799378,
 -0.012305125,
 -0.01497548,
 -0.006503783,
 -0.005624279,
 0.020265369,
 -0.029982245,
 -0.008486281,
 -0.0050998786,
 -0.14985937,
 -0.017222198,
 0.011674307,
 -0.014542328,
 0.02246327,
 -0.009126675,
 -0.018621653,
 -0.004780002,
 0.0020153068,
 -0.0133157885,
 -0.006439805,
 0.0008783784,
 -0.023549458,
 -0.008333907,
 0.01863065,
 -0.0190

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
documents=["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."]

In [18]:
my_query  = "Who is a president of USA?"

In [19]:
embedded_docs = embeddings.embed_documents(documents)
embedded_query = embeddings.embed_query(my_query)

In [20]:
cosine_similarity([embedded_query], embedded_docs)

array([[0.67442095, 0.77875809, 0.59271931]])

In [21]:
from sklearn.metrics.pairwise import euclidean_distances

In [22]:
euclidean_distances([embedded_query], embedded_docs)

array([[0.80694368, 0.66519456, 0.90253057]])

In [26]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

C:\Users\Sunny\AppData\Local\Temp\ipykernel_26728\2965923628.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.in_memory import InMemoryDocstore


In [27]:
len(embeddings.embed_query("Hello world"))

3072

In [28]:
index = faiss.IndexFlatL2(3072)  # 3072 is the dimension of the embeddings

In [29]:
#this is my vector store 
# inside this vectorstrore the data is being store inside the index
# as of now it is inmemory but we can also store it in disk as well
vector_store = FAISS(
    embedding_function = embeddings,
    index = index,
    docstore = InMemoryDocstore(),
    index_to_docstore_id={}
    
)

In [31]:
vector_store.add_texts(["Washington is the capital of USA",
           "Donald Trump is a president of USA",
           "Narendra Modi is a prime minister of India."])

['0777eef0-acc8-4d7b-88b9-664885c06efa',
 '0d422b3c-3f2d-4add-9ebc-db13d621c892',
 '78841699-4feb-4399-b9cb-afcea0664ad3']

In [32]:
vector_store.index_to_docstore_id

{0: '0777eef0-acc8-4d7b-88b9-664885c06efa',
 1: '0d422b3c-3f2d-4add-9ebc-db13d621c892',
 2: '78841699-4feb-4399-b9cb-afcea0664ad3'}

In [33]:
faiss_index_id = 1

In [35]:
docstore_id =vector_store.index_to_docstore_id[faiss_index_id]

In [36]:
vector_store.docstore.search(docstore_id)

Document(id='0d422b3c-3f2d-4add-9ebc-db13d621c892', metadata={}, page_content='Donald Trump is a president of USA')

In [37]:
vector_store.docstore.search(docstore_id).page_content

'Donald Trump is a president of USA'

In [38]:
vector_store.docstore.search(docstore_id).metadata

{}

In [41]:
vector_store.index.reconstruct(faiss_index_id)

array([-0.01705427,  0.01364006,  0.02891811, ...,  0.00777903,
        0.00772134, -0.0054425 ], dtype=float32)

In [42]:
vector_store.index.reconstruct(faiss_index_id).shape

(3072,)

In [39]:
vector_store.similarity_search("Who is a president of USA?", k=1)

[Document(id='0d422b3c-3f2d-4add-9ebc-db13d621c892', metadata={}, page_content='Donald Trump is a president of USA')]

In [40]:
vector_store.similarity_search("Who is a president of USA?", k=2)

[Document(id='0d422b3c-3f2d-4add-9ebc-db13d621c892', metadata={}, page_content='Donald Trump is a president of USA'),
 Document(id='0777eef0-acc8-4d7b-88b9-664885c06efa', metadata={}, page_content='Washington is the capital of USA')]

In [ ]:
laanchain ->documents -> embeddings -> vectorstore -> index -> faiss

In [ ]:
langchain -> everything would be the langchain documents

In [43]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)


In [44]:
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [45]:
vector_store.add_documents(documents=documents)

['85a728eb-c6f5-435a-b8aa-6d789bbd326a',
 '2af8402d-d47d-4233-a061-70557828cf7e',
 'c93a5204-0378-494f-8b65-f53f03af17ce',
 '4ef62265-233c-4ce5-bfbe-123ed0343f24',
 'af6dbfad-22a0-4dc0-ab7e-0d04aaa75c5e',
 'c4c60453-b221-4168-90dd-630b62a6c7d5',
 '80b78c6d-09a8-4258-81a9-4ff8d9e58fdb',
 '553bdbcf-0ec9-4b96-a465-d1da83d14586',
 '56ecbc8b-8eee-47c0-81f9-5f2739351ce6',
 '9b0667d7-d108-426c-8d44-734bbdd72062']

In [47]:
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy",
    k=5)

[Document(id='c93a5204-0378-494f-8b65-f53f03af17ce', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='553bdbcf-0ec9-4b96-a465-d1da83d14586', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='85a728eb-c6f5-435a-b8aa-6d789bbd326a', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(id='56ecbc8b-8eee-47c0-81f9-5f2739351ce6', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='2af8402d-d47d-4233-a061-70557828cf7e', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [49]:
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"})

[Document(id='c93a5204-0378-494f-8b65-f53f03af17ce', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='553bdbcf-0ec9-4b96-a465-d1da83d14586', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='85a728eb-c6f5-435a-b8aa-6d789bbd326a', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.')]

In [50]:
vector_store.similarity_search("LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "news"})

[Document(id='56ecbc8b-8eee-47c0-81f9-5f2739351ce6', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='2af8402d-d47d-4233-a061-70557828cf7e', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='4ef62265-233c-4ce5-bfbe-123ed0343f24', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [ ]:
#disk persistence of the vectorstore
vector_store.save_local("faiss_index")

In [52]:
FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)